In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


# BERT Grievance Severity Classification - Fixed for Local Run

Loads data/raw/Book1.xlsx from local directory, trains BERT classifier on grievance_text -> severity.

In [ ]:
# Install packages if needed (run once)
# !pip install transformers datasets torch accelerate scikit-learn pandas openpyxl -q

: 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

In [ ]:
# Load data - FIXED PATH FOR LOCAL
df = pd.read_excel("data/raw/Book1.xlsx")
print("Columns:", df.columns.tolist())
print(df.head())
print("Severity distribution:")
print(df['severity'].value_counts())

In [ ]:
# Split data
X = df.drop(columns=['severity'])
y = df['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# Label encoding - FIXED
unique_labels = sorted(y_train.unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", unique_labels)

In [ ]:
# Train dataset - FIXED
y_train_encoded = y_train.map(label2id)
train_df = pd.DataFrame({
    "text": X_train['grievance_text'],
    "labels": y_train_encoded
})

train_dataset = Dataset.from_pandas(train_df)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
# Model and training - batch_size reduced for stability
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
# Test dataset and evaluation - FULLY FIXED
y_test_encoded = y_test.map(label2id)
test_df = pd.DataFrame({
    "text": X_test['grievance_text'],
    "labels": y_test_encoded
})

test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Evaluate
results = trainer.evaluate(test_dataset)
print("Evaluation results:", results)

# Metrics
predictions_output = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

accuracy = accuracy_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")